## keras实战

In [2]:
import numpy as np
import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.optimizers import SGD

# 生成虚拟数据
x_train = np.random.random((100,100,100,3))
print(x_train.shape)
y_train = keras.utils.to_categorical(np.random.randint(10, size=(100, 1)), num_classes=10)
x_test = np.random.random((20,100,100,3))
y_test = keras.utils.to_categorical(np.random.randint(10,size=(20,1)),num_classes=10)
print(y_train.shape)

# 构建模型
model = Sequential()

# 添加一个卷积层，32个过滤器，3x3的过滤器大小，ReLU激活函数，输入形状为100x100x3
model.add(Conv2D(32, (3,3),activation='relu',input_shape=(100,100,3)))

# 添加另一个卷积层，32个过滤器，3x3的过滤器大小，ReLU激活函数
model.add(Conv2D(32,(3,3),activation='relu'))

# 添加最大池化层，池化窗口大小为2x2
model.add(MaxPooling2D(pool_size=(2,2)))

# 添加Dropout层，丢弃率为0.25
model.add(Dropout(0.25))

# 添加另一个卷积层，64个过滤器，3x3的过滤器大小，ReLU激活函数
model.add(Conv2D(64,(3,3),activation='relu'))

# 添加另一个卷积层，64个过滤器，3x3的过滤器大小，ReLU激活函数
model.add(Conv2D(64,(3,3),activation='relu'))

# 添加最大池化层，池化窗口大小为2x2
model.add(MaxPooling2D(pool_size=(2,2)))

# 添加Dropout层，丢弃率为0.25
model.add(Dropout(0.25))

# flatten层，将多维输入一维化
model.add(Flatten())

# 添加全连接层，512个神经元，ReLU激活函数
model.add(Dense(512,activation='relu'))

# 添加Dropout层，丢弃率为0.5
model.add(Dropout(0.5))

# 添加全连接
model.add(Dense(10,activation='softmax'))

# 编译模型，使用SGD优化器，学习率为0.01，动量为0.9，损失函数为分类交叉熵，评估指标为准确率
sgd = SGD(lr=0.01,decay=1e-6,momentum=0.9,nesterov=True)

# 编译模型
model.compile(loss='categorical_crossentropy',optimizer=sgd,metrics=['accuracy'])

# 训练模型，批次大小为32，训练10个周期
model.fit(x_train,y_train,batch_size=32,epochs=10)

# 评估模型
score = model.evaluate(x_test,y_test,batch_size=32)

# 输出评估结果
print("Test loss:",score[0])
print("Test accuracy:",score[1])

# 预测
y_predict = model.predict(x_test)
print(y_predict)

(100, 100, 100, 3)
(100, 10)
Epoch 1/10
4/4 [==============================] - 1s 131ms/step - loss: 2.3924 - accuracy: 0.1100
Epoch 2/10
4/4 [==============================] - 1s 129ms/step - loss: 2.3180 - accuracy: 0.1400
Epoch 3/10
4/4 [==============================] - 1s 140ms/step - loss: 2.3033 - accuracy: 0.0900
Epoch 4/10
4/4 [==============================] - 1s 130ms/step - loss: 2.2665 - accuracy: 0.1200
Epoch 5/10
4/4 [==============================] - 1s 129ms/step - loss: 2.2602 - accuracy: 0.1500
Epoch 6/10
4/4 [==============================] - 1s 133ms/step - loss: 2.2501 - accuracy: 0.1100
Epoch 7/10
4/4 [==============================] - 1s 133ms/step - loss: 2.2711 - accuracy: 0.1300
Epoch 8/10
4/4 [==============================] - 1s 135ms/step - loss: 2.2698 - accuracy: 0.1500
Epoch 9/10
4/4 [==============================] - 1s 134ms/step - loss: 2.2736 - accuracy: 0.1200
Epoch 10/10
1/1 [==============================] - 0s 198ms/step - loss: 2.3074 - accurac

## 目标检测
1. 概念：
    - 目标检测（ObjectDetection）的任务是找出图像中所有感兴趣的目标（物体），确定它们的类别和位置，是计算机视觉领域的核心问题之一。由于各类物体有不同的外观，形状，姿态，加上成像时光照，遮挡等因素的干扰，目标检测一直是计算机视觉领域最具有挑战性的问题。
2. 计算机视觉中关于图像识别有四大类任务：
    - **分类-Classification**：解决”是什么？”的问题，即给定一张图片或一段视频判断里面包含什么类别的目标。
    - **定位-Location**：解决“在哪里？”的问题，即定位出这个目标的的位置。
    - **检测-Detection**：解决“是什么？在哪里？”的问题，即定位出这个目标的的位置并且知道目标物是什么。
    - **分割-Segmentation**：分为实例的分割(Instance-level)和场景分割(Scene-level)，解决”每一个像素属于哪个目标物或场景”的问题。
3. 除了图像分类之外，目标检测要解决的核心问题是：
    1. 目标可能出现在图像的任何位置。
    2. 目标有各种不同的大小。
    3. 目标可能有各种不同的形状。
    如果用矩形框来定义目标，则矩形有不同的宽高比。由于目标的宽高比不同，因此采用经典的滑动窗口+图像缩放
    的方案解决通用目标检测问题的成本太高。
4. 基于深度学习的目标检测算法主要分为两类：
    1. Twostage目标检测算法
    先进行区域生成（regionproposal，RP）（一个有可能包含待检物体的预选框），再通过卷积神经网络进行样本
    分类。
    - 任务：特征提取一>生成RP一>分类/定位回归。
    - 常见的two stage目标检测算法有：R-CNN、SPP-Net、Fast R-CNN、Faster R-CNN和R-FCN等。
    2. Onestage目标检测算法
    不用RP，直接在网络中提取特征来预测物体分类和位置。
    - 任务：特征提取一>分类/定位回归。
    - 常见的one stage目标检测算法有：OverFeat、YOLOv1、YOLOv2、YOLOv3、SSD和RetinaNet等。